In [1]:
from pathlib import Path
import h5py
import numpy as np
import pandas as pd
from tqdm import tqdm
from src.module.utils import (
    load_config,
    load_npy,
    process_meanbp,
    get_peak_trough,
    select_strategy,
)
from src.module.imputation import process_impute
from src.module.noise_filtering import process_filter, process_smooth
from src.module.signal_quality_assessment import process_assess

In [2]:
"""
Thoracic: 흉부, 폐, 복부 등의 수술
Cardiac: 심장
Gynecologic: 부인과
Obstetric: 산부인과

VitalDB는 Cardiac & Obstetric surgery는 애초에 배제하고 수집되었으며
MOVER는 ICD-10-PCS 코드 중 앞 두자리가 다음과 같은 경우를 제외하면 되겠다:
- Cardiac = '02*'
- Obstetric = '10*'

[최종] Cardiac & Gynecologic surgery 배제
VitalDB = Exclude cardiac, gynecologic
MOVER = Exclude cardiac('02'), gynecologic('10', '0U')
"""


def vitaldb_prepare_cohort(file_path) -> pd.DataFrame:
    cohort = pd.read_csv(file_path / "clinical_data.csv").reset_index(drop=True)
    cohort["age_num"] = np.where((cohort["age"] == ">89"), 90, cohort["age"]).astype(
        np.float32  # age < 1
    )
    cohort = (
        cohort.query("18 <= age_num")
        .drop_duplicates(["caseid"], keep=False)
        .query("department not in ['Gynecology']")
        # .query("department not in ['Thoracic surgery', 'Gynecology']")
        .query("ane_type == 'General'")
    )
    return cohort


def mover_prepare_cohort(file_path) -> pd.DataFrame:
    pat_info = pd.read_csv(file_path / "patient_information.csv")
    pat_code = pd.read_csv(file_path / "patient_coding.csv")
    pid_to_mrn = pd.read_csv(file_path / "EPIC_MRN_PAT_ID.csv")
    selected_mrn = pat_code[
        ~pat_code.REF_BILL_CODE.apply(lambda x: str(x)[:2] in ["02", "10", "0U"])
    ]["MRN"].unique()
    cohort = (
        pd.merge(pid_to_mrn, pat_info, how="inner", on=["LOG_ID", "MRN"])
        .drop_duplicates()
        .reset_index(drop=True)
    )
    cohort = (
        cohort.query("18 <= BIRTH_DATE")
        .query("PRIMARY_ANES_TYPE_NM == 'General'")
        .query("MRN.isin(@selected_mrn)")  # cardiac & obstetric surgery excluded
        .reset_index(drop=True)
    )
    return cohort

In [3]:
data_dir = Path("../../data")
vitaldb_cohort = vitaldb_prepare_cohort(data_dir / "vitaldb")
mover_cohort = mover_prepare_cohort(data_dir / "mover")

In [4]:
# # 전처리된 데이터만 baseline characterisitcs를 추출한다.
# vitaldb_caselist = sorted(
#     [
#         int(x.parent.name)
#         for x in (data_dir / "vitaldb" / "02_04.processed").rglob("*.h5")
#     ]
# )
# mover_caselist = sorted(
#     [x.parent.name for x in (data_dir / "mover" / "02.processed").rglob("*.h5")]
# )
# print(len(vitaldb_caselist), len(mover_caselist))

# # 주의!
# # vitaldb에서 수집된 caselist는 환자 단위가 아니다.
# # mover에서 수집된 caselist는 환자 단위이다.

In [5]:
config = load_config("../../config/conf.yaml")
src_path = Path(config["data_path"]).expanduser()

In [6]:
vdb_pos_cid = np.hstack(
    [
        load_npy(fp)
        for fp in (src_path / "vitaldb" / "05.dataset_new").rglob(
            "hypophetversion2*pos_cid.npy"
        )
    ]
)
vdb_neg_cid = np.hstack(
    [
        load_npy(fp)
        for fp in (src_path / "vitaldb" / "05.dataset_new").rglob(
            "hypophetversion2*neg_cid.npy"
        )
    ]
)
vitaldb_caselist = np.unique(np.hstack([vdb_pos_cid, vdb_neg_cid])).astype(int)
print(f"case no: {vitaldb_caselist.size}")

case no: 3247


In [7]:
mov_pos_cid = np.hstack(
    [
        load_npy(fp)
        for fp in (src_path / "mover" / "05.dataset_new").rglob(
            "hypophetversion2*pos_cid.npy"
        )
    ]
)
mov_neg_cid = np.hstack(
    [
        load_npy(fp)
        for fp in (src_path / "mover" / "05.dataset_new").rglob(
            "hypophetversion2*neg_cid.npy"
        )
    ]
)
mover_caselist = np.unique(np.hstack([mov_pos_cid, mov_neg_cid]))
print(f"case no: {mover_caselist.size}")

case no: 649


In [8]:
vitaldb_cohort = vitaldb_cohort.query("caseid.isin(@vitaldb_caselist)").reset_index(
    drop=True
)
print("VitalDB case no: ", vitaldb_cohort.caseid.nunique())
print("VitalDB patient no: ", vitaldb_cohort.subjectid.nunique())

mover_cohort = mover_cohort.query("PAT_ID.isin(@mover_caselist)").reset_index(drop=True)
print("MOVER case no: ", mover_cohort.LOG_ID.nunique())
print("MOVER patient no: ", mover_cohort.PAT_ID.nunique())

VitalDB case no:  3247
VitalDB patient no:  3125
MOVER case no:  1199
MOVER patient no:  649


In [9]:
config = load_config("../../config/conf.yaml")
fs = config["fs"]

In [10]:
src_path = Path(config["data_path"]).expanduser()
v_path = vitaldb_cohort.caseid.apply(
    lambda x: str(src_path / "vitaldb" / "01.raw" / str(x).zfill(4) / str(x).zfill(4))
    + "_100fs.h5"
).values
m_path = mover_cohort.PAT_ID.apply(
    lambda x: str(src_path / "mover" / "01.raw" / x / x) + "_100fs.h5"
).values

# Missingness

MOVER는 애초에 nan이 없으므로 gap을 확인할 필요가 없다. (`01.baseline_characteristics.ipynb` 참고.)

In [11]:
vmiss = []
for file_path in tqdm(v_path, ncols=75):
    with h5py.File(file_path, "r") as f:
        sig = f["signal/abp"][:].reshape(-1)
    vmiss.append(np.isnan(sig).sum() / len(sig))
vmiss = np.array(vmiss)
print(
    f"VitalDB: {np.nanmedian(vmiss)} [{np.quantile(vmiss, 0.25)}, {np.quantile(vmiss, 0.75)}]"
)
print("<0.01%")

100%|██████████████████████████████████| 3247/3247 [01:07<00:00, 47.94it/s]

VitalDB: 8.585707067517211e-05 [0.0, 0.0013958110220039126]
<0.01%


# Gap

In [12]:
sig_list = []
for file_path in tqdm(v_path, ncols=75):
    with h5py.File(file_path, "r") as f:
        sig = f["signal/abp"][:].reshape(-1)
    if np.isnan(sig).sum() == 0:
        continue
    sig_list.append(sig)

100%|██████████████████████████████████| 3247/3247 [01:17<00:00, 41.96it/s]


In [15]:
gap_list = []
for sig in tqdm(sig_list, ncols=75):
    gap_list.append(
        (np.where(np.diff(np.isnan(sig), prepend=False) != 0)[0]).reshape(-1, 2)
    )
gaps = np.diff(np.vstack(gap_list), axis=1)

100%|████████████████████████████████| 2012/2012 [00:01<00:00, 1217.77it/s]


In [26]:
print(
    f"Missing gap (sec): {np.percentile(gaps, 50) / fs} [{np.percentile(gaps, 25) / fs}, {np.percentile(gaps, 75) / fs}] ({np.percentile(gaps, 0) / fs}, {np.percentile(gaps, 100) / fs})"
)

Missing gap (sec): 1.99 [0.2, 18.22] (0.01, 15864.9)


# Segments (total, excluded, eligible)

In [ ]:
def range_segment(segments):
    if segments.size != 0:
        ranges = np.apply_along_axis(lambda x: np.arange(*x), axis=1, arr=segments)
    else:
        ranges = np.array([], dtype=np.int32)
    return ranges


def assess_segment(ranges, sqi, config):
    if len(ranges) != 0:
        is_bad_quality = np.any(sqi[ranges] > config["sqi"]["threshold"], axis=1)
        samples = ranges[~is_bad_quality]
    else:
        samples = np.array([], dtype=np.int32).reshape(
            -1, config["fs"] * config["input_len"]
        )
    return samples

In [12]:
config = load_config("../../config/conf.yaml")
src_path = Path(config["data_path"]).expanduser()
strategy = select_strategy("hypophetversion2")

HypophetVersion2 strategy selected.


In [40]:
vdb_cohort_cid = vitaldb_cohort.caseid.apply(lambda x: str(x).zfill(4)).values
sample_path = src_path / "vitaldb" / "02_04.processed"
pos_event_path = sorted(sample_path.rglob(f"*{strategy.name}*_pred_pos_events.npy"))
pos_event_path = [x for x in pos_event_path if not x.name.startswith("._")]
pos_event_path = sorted(
    filter(lambda x: x.parent.name in vdb_cohort_cid, pos_event_path)
)
neg_event_path = sorted(sample_path.rglob(f"*{strategy.name}*_neg_events.npy"))
neg_event_path = [x for x in neg_event_path if not x.name.startswith("._")]
neg_event_path = sorted(
    filter(lambda x: x.parent.name in vdb_cohort_cid, neg_event_path)
)

In [41]:
pos_segments, pos_eligible = [], []
for event_path in tqdm(pos_event_path, ncols=75):
    cid = str(event_path).split("/")[-2]
    case_path = sample_path / cid
    sig_path = case_path / f"{cid}_100fs_processed.h5"
    with h5py.File(sig_path, "r") as f:
        sqi = f["signal/sqi"][:].astype(np.float32)
    events = load_npy(event_path)
    segments = strategy.extract_segment(events, config, "pos")
    ranges = range_segment(segments)
    samples = assess_segment(ranges, sqi, config)
    pos_segments.append(segments)
    pos_eligible.append(samples)

pos_segments = np.vstack([seg for seg in pos_segments if len(seg) != 0])
pos_eligible = np.vstack([seg for seg in pos_eligible if len(seg) != 0])
print(f"""
Segments
* Total:\t{len(pos_segments)}
* Excluded:\t{len(pos_segments) - len(pos_eligible)}
* Eligible:\t{len(pos_eligible)}
""")

100%|█████████████████████████████████| 3247/3247 [00:21<00:00, 153.02it/s]


Segments
* Total:	116645
* Excluded:	37617
* Eligible:	79028



In [42]:
neg_segments, neg_eligible = [], []
for event_path in tqdm(neg_event_path, ncols=75):
    cid = str(event_path).split("/")[-2]
    case_path = sample_path / cid
    sig_path = case_path / f"{cid}_100fs_processed.h5"
    with h5py.File(sig_path, "r") as f:
        sqi = f["signal/sqi"][:].astype(np.float32)
    events = load_npy(event_path)
    segments = strategy.extract_segment(events, config, "neg")
    ranges = range_segment(segments)
    samples = assess_segment(ranges, sqi, config)
    neg_segments.append(segments)
    neg_eligible.append(samples)

neg_segments = np.vstack([seg for seg in neg_segments if len(seg) != 0])
neg_eligible = np.vstack([seg for seg in neg_eligible if len(seg) != 0])
print(f"""
Segments
* Total:\t{len(neg_segments)}
* Excluded:\t{len(neg_segments) - len(neg_eligible)}
* Eligible:\t{len(neg_eligible)}
""")

100%|█████████████████████████████████| 3247/3247 [00:19<00:00, 166.43it/s]



Segments
* Total:	204927
* Excluded:	56637
* Eligible:	148290



In [43]:
vdb_segments = np.vstack([pos_segments, neg_segments])
vdb_eligible = np.vstack([pos_eligible, neg_eligible])
print(f"""
Segments
* Total:\t{len(vdb_segments)}
* Excluded:\t{len(vdb_segments) - len(vdb_eligible)}
* Eligible:\t{len(vdb_eligible)}
""")


Segments
* Total:	321572
* Excluded:	94254
* Eligible:	227318



In [54]:
mover_cohort_cid = mover_cohort.PAT_ID.values
sample_path = src_path / "mover" / "03.label"
pos_event_path = sorted(sample_path.rglob(f"*{strategy.name}*_pred_pos_events.npy"))
pos_event_path = [x for x in pos_event_path if not x.name.startswith("._")]
pos_event_path = sorted(
    filter(lambda x: x.parent.name in mover_cohort_cid, pos_event_path)
)
neg_event_path = sorted(sample_path.rglob(f"*{strategy.name}*_neg_events.npy"))
neg_event_path = [x for x in neg_event_path if not x.name.startswith("._")]
neg_event_path = sorted(
    filter(lambda x: x.parent.name in mover_cohort_cid, neg_event_path)
)

In [57]:
pos_segments, pos_eligible = [], []
for event_path in tqdm(pos_event_path, ncols=75):
    cid = str(event_path).split("/")[-2]
    sig_path = src_path / "mover" / "02.processed" / cid / f"{cid}_100fs_processed.h5"
    with h5py.File(sig_path, "r") as f:
        sqi = f["signal/sqi"][:].astype(np.float32)
    events = load_npy(event_path)
    segments = strategy.extract_segment(events, config, "pos")
    ranges = range_segment(segments)
    samples = assess_segment(ranges, sqi, config)
    pos_segments.append(segments)
    pos_eligible.append(samples)

pos_segments = np.vstack([seg for seg in pos_segments if len(seg) != 0])
pos_eligible = np.vstack([seg for seg in pos_eligible if len(seg) != 0])
print(f"""
Segments
* Total:\t{len(pos_segments)}
* Excluded:\t{len(pos_segments) - len(pos_eligible)}
* Eligible:\t{len(pos_eligible)}
""")

100%|████████████████████████████████████| 649/649 [00:53<00:00, 12.17it/s]


Segments
* Total:	19010
* Excluded:	9715
* Eligible:	9295



In [58]:
neg_segments, neg_eligible = [], []
for event_path in tqdm(neg_event_path, ncols=75):
    cid = str(event_path).split("/")[-2]
    sig_path = src_path / "mover" / "02.processed" / cid / f"{cid}_100fs_processed.h5"
    with h5py.File(sig_path, "r") as f:
        sqi = f["signal/sqi"][:].astype(np.float32)
    events = load_npy(event_path)
    segments = strategy.extract_segment(events, config, "neg")
    ranges = range_segment(segments)
    samples = assess_segment(ranges, sqi, config)
    neg_segments.append(segments)
    neg_eligible.append(samples)

neg_segments = np.vstack([seg for seg in neg_segments if len(seg) != 0])
neg_eligible = np.vstack([seg for seg in neg_eligible if len(seg) != 0])
print(f"""
Segments
* Total:\t{len(neg_segments)}
* Excluded:\t{len(neg_segments) - len(neg_eligible)}
* Eligible:\t{len(neg_eligible)}
""")

100%|████████████████████████████████████| 649/649 [00:11<00:00, 57.96it/s]



Segments
* Total:	90333
* Excluded:	73835
* Eligible:	16498



In [61]:
vdb_pos_seg = np.hstack(
    [
        load_npy(fp)
        for fp in (src_path / "vitaldb" / "05.dataset_new").rglob(
            "hypophetversion2*pos_segments.npy"
        )
    ]
)
vdb_neg_seg = np.hstack(
    [
        load_npy(fp)
        for fp in (src_path / "vitaldb" / "05.dataset_new").rglob(
            "hypophetversion2*neg_segments.npy"
        )
    ]
)
vdb_seg = np.vstack([vdb_pos_seg, vdb_neg_seg])
print(vdb_pos_seg.shape, vdb_neg_seg.shape, vdb_seg.shape)

(79028, 3000) (148290, 3000) (227318, 3000)


In [59]:
mov_segments = np.vstack([pos_segments, neg_segments])
mov_eligible = np.vstack([pos_eligible, neg_eligible])
print(f"""
Segments
* Total:\t{len(mov_segments)}
* Excluded:\t{len(mov_segments) - len(mov_eligible)}
* Eligible:\t{len(mov_eligible)}
""")


Segments
* Total:	109343
* Excluded:	83550
* Eligible:	25793



In [60]:
mov_pos_seg = np.hstack(
    [
        load_npy(fp)
        for fp in (src_path / "mover" / "05.dataset_new").rglob(
            "hypophetversion2*pos_segments.npy"
        )
    ]
)
mov_neg_seg = np.hstack(
    [
        load_npy(fp)
        for fp in (src_path / "mover" / "05.dataset_new").rglob(
            "hypophetversion2*neg_segments.npy"
        )
    ]
)
mov_seg = np.vstack([mov_pos_seg, mov_neg_seg])
print(mov_pos_seg.shape, mov_neg_seg.shape, mov_seg.shape)

(9295, 3000) (16498, 3000) (25793, 3000)
